In [ ]:
import os
from dotenv import load_dotenv
import databento as db

load_dotenv()
db_api_key = os.getenv("DATABENTO_API_KEY")

client = db.Historical(db_api_key)

try:
    cost = client.metadata.get_cost(
        dataset="GLBX.MDP3",
        schema="ohlcv-1m",
        stype_in="continuous",
        symbols=["NQ.v.0"],
        start="2026-11-17",
        end="2026-11-26",
    )
    print(cost, 'USD')
except db.BentoClientError as e:
    print(f"Error: {e}")


In [ ]:
try:
    db_data = client.timeseries.get_range(
        dataset="GLBX.MDP3",
        schema="ohlcv-1m",
        stype_in="continuous",
        symbols=["NQ.v.0"],
        start="2026-11-17",
        end="2026-11-26",
    )
    db_data = db_data.to_df()
except db.BentoClientError as e:
    print(f"Error: {e}")

- Fact: Databento flags 2026-11-22 as degraded
- Impact: store a quality flag per day, check if the model is sensitive to it

In [ ]:
display(db_data.head())
display(db_data.columns)
display(db_data.dtypes) 
display(db_data.index.tz) 
display(len(db_data))

- Fact: 9 columns, prices are float64, index in UTC
- Impact: keep prices, volume, index, instrument_id drop symbols, rtype, publisher_id
- Fact: 9 660 rows, expected 9 660 rows
- Impact: don't assume every minute exists (holidays, short sessions); add a dbt test comparing expected vs received bars per day

In [ ]:
db_ny = db_data.tz_convert("America/New_York")
db_ny.head()

In [ ]:
display(db_ny.groupby(db_ny.index.hour).size())
display(db_ny.groupby(db_ny.index.day_of_week).size())

- Fact: no bars between 17:00 and 18:00 New York time (daily halt), no Saturday, Sunday only from 18:00
- Fact: bar counts per calendar day are uneven, because a NQ session runs from 18:00 the previous day to 17:00 (New York time)
- Impact: add a `session_date` column following the CME rule (a session is named after the day it ends), never group by calendar date
- Impact: always convert to New York time with pandas (`tz_convert`), never with a fixed offset, because of daylight saving time

- Fact: the 9:30-16:00 window is the regular session (RTH) of the cash market; NQ trades almost 24h on Globex
- Impact: organise data by Globex session, compute the target and the gap on the RTH window

In [ ]:
display(db_ny.groupby(db_ny.index.hour)["volume"].sum())

volume_by_hour = db_ny.groupby(db_ny.index.hour)["volume"].sum()
rth_share = volume_by_hour.loc[9:15].sum() / volume_by_hour.sum()
rth_share

- Fact: ~77% of the volume trades between 9:00 and 16:00 New York time, peak at 10:00 (~15x an average night hour)
- Fact: volume jumps at 8:00, when US macro data is released (8:30)
- Impact: pre-market features come from low-volume hours, so they are noisier; prefer ranges and returns over single prices
- Impact: macro release days (8:30) need a flag, they change the pre-market behaviour